In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7" 

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader
from vllm import LLM, SamplingParams

In [2]:
model_path = "/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise/actor/global_step_150"

tokenizer = AutoTokenizer.from_pretrained(model_path)
torch.cuda.empty_cache()
base_model = LLM(
    model=model_path,
    tensor_parallel_size=1,  # 使用全部8张GPU
    gpu_memory_utilization=0.85,  # 可以设置更高的内存利用率
    dtype="auto"
)

INFO 08-18 10:37:59 config.py:1450] Downcasting torch.float32 to torch.float16.
INFO 08-18 10:37:59 llm_engine.py:174] Initializing an LLM engine (v0.5.4) with config: model='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise/actor/global_step_150', speculative_config=None, tokenizer='/mnt/petrelfs/huzican/R1/rlvr_div/checkpoints/div/pos_rs_0.1-neg_rs_0.01-equ+belu+rule-piecewise/actor/global_step_150', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_tr

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]


INFO 08-18 10:38:06 model_runner.py:732] Loading model weights took 14.2448 GB
INFO 08-18 10:38:07 gpu_executor.py:102] # GPU blocks: 59353, # CPU blocks: 4681
INFO 08-18 10:38:10 model_runner.py:1024] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 08-18 10:38:10 model_runner.py:1028] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 08-18 10:38:22 model_runner.py:1225] Graph capturing finished in 12 secs.


In [3]:
val_data_path = "dataset/aime.parquet"
val_dataset = RLHFDataset(parquet_files=val_data_path,
                            tokenizer=tokenizer,
                            prompt_key='prompt',
                            max_prompt_length=1024,
                            filter_prompts=True,
                            return_raw_chat=False,
                            truncation='error')

val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=8,
                            shuffle=True,
                            drop_last=True,
                            collate_fn=collate_fn)

original dataset len: 30
filter dataset len: 30


In [4]:
pad_token_id = tokenizer.pad_token_id
from typing import List
def _pre_process_inputs(pad_token_id, prompt_token_ids: torch.Tensor) -> List[int]:
    # remove the left padding in the prompt token_id
    # pad_token_id = self.llm_engine.tokenizer.pad_token_id if self.llm_engine.tokenizer.pad_token_id is not None else self.llm_engine.tokenizer.eos_token_id
    non_pad_index = torch.nonzero(prompt_token_ids != pad_token_id, as_tuple=False)[0][0]
    token_ids = prompt_token_ids[non_pad_index:].tolist()
    return token_ids


In [5]:
reward_tensor_lst = []
data_source_lst = []
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=1.0,
    max_tokens=8192,
)
for test_data in val_dataloader:
    # print(test_data)
    
    idx = test_data['input_ids']
    batch_size = idx.size(0)
    idx_list = [_pre_process_inputs(pad_token_id, idx[i]) for i in range(batch_size)]
    output = base_model.generate(prompts=None,
                             sampling_params=sampling_params,
                             prompt_token_ids=idx_list,
                             use_tqdm=True)
    print(output)
    break

    

Processed prompts: 100%|██████████| 8/8 [01:49<00:00, 13.73s/it, est. speed input: 19.69 toks/s, output: 421.99 toks/s]

[RequestOutput(request_id=0, prompt=None, prompt_token_ids=[7771, 3383, 374, 311, 1795, 264, 36438, 11, 17423, 32711, 1882, 1573, 8241, 279, 1590, 6291, 13, 1096, 17601, 41018, 11, 28285, 4849, 11, 23966, 11, 31734, 433, 287, 11, 323, 73185, 697, 3381, 1882, 1526, 5248, 25687, 13, 28596, 697, 2033, 1119, 1378, 14158, 25, 35187, 323, 12478, 13, 758, 279, 35187, 3772, 11, 3042, 697, 32711, 1667, 279, 3561, 25, 1036, 27, 26865, 397, 314, 60565, 82, 92, 690, 26865, 397, 11204, 8886, 3381, 1265, 2924, 11682, 6358, 11, 86781, 287, 11, 22901, 11, 323, 72913, 315, 6708, 13, 4636, 1036, 522, 26865, 397, 2419, 304, 279, 12478, 3772, 11, 3410, 279, 1590, 11, 19819, 11, 323, 13382, 4226, 11, 9355, 14257, 504, 279, 26403, 304, 279, 35187, 3772, 13, 1416, 8415, 11, 2924, 279, 4226, 304, 1124, 79075, 6257, 369, 7877, 8460, 3059, 1075, 5248, 11454, 476, 35972, 9904, 13, 2657, 25, 1096, 374, 279, 3491, 510, 9885, 279, 7772, 3204, 1931, 949, 315, 1124, 9697, 22, 20, 10, 16, 16, 22, 72, 8, 89, 41715, 370

In [6]:
output[0].outputs[0].text

"First, let's represent $z$ in its polar form. We know that $|z|=4$, so let $z = 4e^{i\\theta} = 4(\\cos \\theta + i\\sin \\theta)$. Now, we can rewrite the given expression:\n\n\\[\n(75 + 117i)z + \\frac{96 + 144i}{z} = (75 + 117i) \\cdot 4(\\cos \\theta + i\\sin \\theta) + \\frac{96 + 144i}{4(\\cos \\theta + i\\sin \\theta)}.\n\\]\n\nLet's simplify the second term:\n\n\\[\n\\frac{96 + 144i}{4(\\cos \\theta + i\\sin \\theta)} = \\frac{96 + 144i}{4} \\cdot \\frac{1}{\\cos \\theta + i\\sin \\theta} = (24 + 36i) \\cdot \\frac{\\cos \\theta - i\\sin \\theta}{\\cos^2 \\theta + \\sin^2 \\theta} = (24 + 36i)(\\cos \\theta - i\\sin \\theta).\n\\]\n\nThis is because $\\frac{1}{\\cos \\theta + i\\sin \\theta} = \\cos \\theta - i\\sin \\theta$ (since $|\\cos \\theta + i\\sin \\theta| = 1$).\n\nSo, the expression becomes:\n\n\\[\n(75 + 117i) \\cdot 4(\\cos \\theta + i\\sin \\theta) + (24 + 36i)(\\cos \\theta - i\\sin \\theta).\n\\]\n\nExpanding both terms, we have:\n\n\\[\n4(75\\cos \\theta + 75i